## Scryfall Artwork MDS Model

In [1]:
import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ast import literal_eval

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn import manifold
from sklearn.metrics import pairwise_distances

In [2]:
df = pd.read_csv('./data/final-scryfall-unique-artwork.csv')
creature_df = df[df['type'].str.contains('Creature')]
creature_df.head()

,id,oracle_id,name,set_id,set,set_name,artist_ids,artist,released_at,type_line,...,power,toughness,edhrec_rank,type,subtype,legality_commander,legality_standard,price_usd,image_uri_normal,image_uri_art_crop
1,0000579f-7b35-4ed3-b44c-db2a538066fe,44623693-51d6-49ad-8cd7-140505caf02f,Fury Sliver,c1d109bc-ffd8-428f-8d7d-3f8d7e648046,tsp,Time Spiral,['d48dd097-720d-476a-8722-6a02854ae28b'],Paolo Parente,2006-10-06,Creature — Sliver,...,3,3,9808.0,Creature,Sliver,legal,not_legal,0.46,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
2,00006596-1166-4a79-8443-ca9f82e6db4e,8ae3562f-28b7-4462-96ed-be0cf7052ccc,Kor Outfitter,eb16a2bd-a218-4e4e-8339-4aa1afc0c8d2,zen,Zendikar,['aa7e89ed-d294-4633-9057-ce04dacfcfa4'],Kieran Yanner,2009-10-02,Creature — Kor Soldier,...,2,2,19672.0,Creature,Kor Soldier,legal,not_legal,0.11,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
3,0000cd57-91fe-411f-b798-646e965eec37,9f0d82ae-38bf-45d8-8cda-982b6ead1d72,Siren Lookout,fe0dad85-54bc-4151-9200-d68da84dd0f2,xln,Ixalan,['a8e7b854-b15a-421a-b66d-6e68187ae285'],Chris Rallis,2017-09-29,Creature — Siren Pirate,...,1,2,18843.0,Creature,Siren Pirate,legal,not_legal,0.04,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
4,0001f1ef-b957-4a55-b47f-14839cdbab6f,ef027846-be81-4959-a6b5-56bd01b1e68a,Venerable Knight,a90a7b2f-9dd8-4fc7-9f7d-8ea2797ec782,eld,Throne of Eldraine,['9c201dbe-db56-429a-87e6-189ea70c2632'],Colin Boyer,2019-10-04,Creature — Human Knight,...,2,1,18404.0,Creature,Human Knight,legal,not_legal,0.15,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
6,0002ab72-834b-4c81-82b1-0d2760ea96b0,645b5784-a6f7-4cf3-966a-e1a51420b96b,Mystic Skyfish,bc94aba1-7376-4e02-a12d-3a2efb66ab0f,m21,Core Set 2021,['bb677b1a-ce51-4888-83d6-5a94de461ff9'],Alayna Danner,2020-07-03,Creature — Fish,...,3,1,23732.0,Creature,Fish,legal,not_legal,0.10,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...


In [3]:
print(len(creature_df))

20408


In [4]:
creature_df['price_usd'].describe()

count    20408.000000
mean         5.292274
std         86.788365
min          0.010000
25%          0.090000
50%          0.200000
75%          0.710000
max       7618.212000
Name: price_usd, dtype: float64

In [5]:
# calculate thresholds for top 1%, bottom 1%, and middle 1% by price
creature_df.loc[:,'price_usd'] = creature_df['price_usd'].round(2)

top_1 = creature_df['price_usd'].quantile(0.99)
bottom_1 = creature_df['price_usd'].quantile(0.01)
mid_lower = creature_df['price_usd'].quantile(0.495) # just below median
mid_upper = creature_df['price_usd'].quantile(0.505) # just above median

top_1_df = creature_df[creature_df['price_usd'] >= top_1]
top_1_df['percentile'] = 'Top 1%'

bottom_1_df = creature_df[creature_df['price_usd'] <= bottom_1]
bottom_1_df['percentile'] = 'Bottom 1%'

mid_1_df = creature_df[(creature_df['price_usd'] >= mid_lower) & (creature_df['price_usd'] < mid_upper)]
mid_1_df['percentile'] = 'Middle 1%'

percentile_df = pd.concat([top_1_df, bottom_1_df, mid_1_df]).reset_index()
print(len(percentile_df))


874


### One-Hot Encoding

Using `MultiLabelBinarizer` since the cards can have multiple keywords (e.g. ['Flying', 'Trample']) and color identities (e.g. ['B', 'W']). The same applies for the subtype - if the subtype is 'Human Knight', we will create the encoding as Human-1, Knight-1. This way, human creatures and knight creatures will cluster, rather than the distinct human knights.

In [6]:
one_hot_encoding_df = percentile_df.copy()
one_hot_encoding_df['subtype'] = percentile_df['subtype'].fillna('').str.split()

In [7]:
mlb = MultiLabelBinarizer()
encoded_subtypes = mlb.fit_transform(one_hot_encoding_df['subtype'])
subtypes_df = pd.DataFrame(encoded_subtypes, columns='s_'+mlb.classes_, index=one_hot_encoding_df.index)
subtypes_df.head()

,s_Advisor,s_Aetherborn,s_Ally,s_Angel,s_Antelope,s_Ape,s_Archer,s_Artificer,s_Assassin,s_Avatar,...,s_Warlock,s_Warrior,s_Whale,s_Wizard,s_Wolf,s_Wolverine,s_Wraith,s_Wurm,s_Yeti,s_Zombie
0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
encoded_keywords = mlb.fit_transform(one_hot_encoding_df['keywords'].apply(literal_eval))
keywords_df = pd.DataFrame(encoded_keywords, columns='k_'+mlb.classes_, index=one_hot_encoding_df.index)
keywords_df.head()

,k_Adapt,k_Affinity,k_Afterlife,k_Alluring Eyes,k_Amass,k_Ascend,k_Assemble,k_Assist,k_Banding,k_Battle Cry,...,k_Treasure,k_Typecycling,k_Undying,k_Unleash,k_Venture into the dungeon,k_Vigilance,k_Void,k_Ward,k_Warp,k_Will of the council
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
encoded_colors = mlb.fit_transform(one_hot_encoding_df['color_identity'].apply(literal_eval))
colors_df = pd.DataFrame(encoded_colors, columns='c_'+mlb.classes_, index=one_hot_encoding_df.index)
colors_df.head()

,c_B,c_G,c_R,c_U,c_W
0,0,1,0,0,0
1,0,1,0,0,0
2,1,1,1,1,1
3,0,0,1,0,0
4,0,0,0,1,0


In [10]:
features = pd.concat([subtypes_df, keywords_df, colors_df], axis=1)
features.head()

,s_Advisor,s_Aetherborn,s_Ally,s_Angel,s_Antelope,s_Ape,s_Archer,s_Artificer,s_Assassin,s_Avatar,...,k_Vigilance,k_Void,k_Ward,k_Warp,k_Will of the council,c_B,c_G,c_R,c_U,c_W
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,1,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


### MDS

In [11]:
# calculate dissimilarity (distance) matrix since data is binary/categorical
distance_matrix = pairwise_distances(features.to_numpy(), metric='jaccard') 

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/pairwise.py:2459: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [12]:
seed = np.random.RandomState(seed=42)
best_stress = float('inf')
best_model = None

for max_iter in [1000, 3000, 5000]:
    for eps in [1e-6, 1e-9]:
        mds = manifold.MDS(n_components=2, metric='precomputed', max_iter=max_iter, eps=eps,
                           normalized_stress=True, n_init=1, init='classical_mds', random_state=seed)
        pos = mds.fit_transform(distance_matrix)
        
        if mds.stress_ < best_stress:
            best_stress = mds.stress_
            best_model = mds

print(f"Best Stress: {best_stress}")
print(f"Best Parameters: {best_model.get_params()}")

Best Stress: 0.3887583775740069
Best Parameters: {'dissimilarity': 'deprecated', 'eps': 1e-09, 'init': 'classical_mds', 'max_iter': 3000, 'metric': 'precomputed', 'metric_mds': True, 'metric_params': None, 'n_components': 2, 'n_init': 1, 'n_jobs': None, 'normalized_stress': True, 'random_state': RandomState(MT19937) at 0x133764440, 'verbose': 0}


In [13]:
pos = best_model.fit_transform(distance_matrix)
percentile_df['x'] = [x[0] for x in pos]
percentile_df['y'] = [x[1] for x in pos]

In [14]:
def genMDSPlot(df, sample_n=200, art_crop=True):
    # input: df -- dataframe (augmented with the x/y columns)
    # input: sample_n -- number of cards to display from each percentile, randomly sampled
    # input: art_crop -- True to use cropped art image, False to use full card image
    # return: an altair chart (e.g., return alt.Chart(...))

    sampled_df = df.groupby('percentile').apply(lambda x: x.sample(n=sample_n, random_state=42)).reset_index()

    image_url = 'image_uri_art_crop' if art_crop else 'image_uri_normal'
    
    images = alt.Chart(sampled_df).mark_image(width=40, height=40).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        url=image_url,
        tooltip=[
            alt.Tooltip("name"),
            alt.Tooltip("subtype"),
            alt.Tooltip("keywords"),
            alt.Tooltip("color_identity"),
            alt.Tooltip("price_usd")
        ]
    ).properties(
        width=1000,
        height=1000
    )

    squares = alt.Chart(sampled_df).mark_square(size=2000).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        color=alt.Color('percentile:N', legend=alt.Legend(title="Percentile"))
    )

    return squares + images

In [15]:
chart = genMDSPlot(percentile_df, 100)
chart

alt.LayerChart(...)

In [16]:
# save interactive chart as HTML
chart.save('artwork-cluster-vis.html')

In [17]:
# show only 50 cards in each percentile, with full card image
genMDSPlot(percentile_df, 50, art_crop=False)

alt.LayerChart(...)